# Handling class imbalance with Ensemble Learning
In this experiment, our primary focus is on mitigating the issue of class imbalance in the training data. 
Specifically, we compare the performance of Decision Three, Random Forest, Gradient Boosting and XGBoost classifiers on a synthetic dataset, whose 80% of the training samples belong to one class.

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from xgboost import XGBClassifier
import numpy as np
import matplotlib.pyplot as plt
import warnings

SEED=42

warnings.filterwarnings("ignore")

## Create imbalanced dataset

In [ ]:
X, y = make_classification(
    n_samples=20000,
    n_features=15,
    n_informative=10,
    n_redundant=4,
    weights=[0.80, 0.20],
    flip_y=0.01,
    class_sep=1.0,
    random_state=SEED
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, stratify=y_train, test_size=0.2, random_state=42
)

# Compute imbalance ratio for XGBoost
ratio = np.sum(y == 0) / np.sum(y == 1)
print(f"Imbalance ratio (majority/minority): {ratio:.1f}")

## Define models

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(min_samples_leaf=10, random_state=SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=1000,
        min_samples_leaf=10,
        oob_score=True,
        random_state=SEED,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=1000,
        learning_rate=0.1,
        min_samples_leaf=10,
        random_state=SEED
    ),
    "XGBoost": XGBClassifier(
        n_estimators=1000,
        learning_rate=0.1,
        min_samples_leaf=10,
        random_state=SEED,
        early_stopping_rounds=10,
        n_jobs=-1
    )
}

## Train, evaluate and plot confusion matrices

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(13, 4))

for ax, (name, model) in zip(axes, models.items()):
    if name == "XGBoost":
        model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],     # validation data
        verbose=False                   # optional: see progress
    )
    else:
        model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, digits=3))
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()